# Non-paired restore notebook

Reads a backup manifest from secondary-region storage and restores the lakehouse backup into the recovered DR workspace.

In [ ]:
import json

try:
    from notebookutils import fs as fabric_fs
except ImportError:
    try:
        from mssparkutils import fs as fabric_fs
    except ImportError:
        fabric_fs = None

if fabric_fs is None:
    raise RuntimeError('This notebook must run in Fabric or Synapse with notebookutils/mssparkutils.fs available.')

backup_manifest_path = 'abfss://fabric-exports@storageaccount.dfs.core.windows.net/fabric-bcdr-demo/source-workspace/source-lakehouse/20260101T000000Z/backup-manifest.json'
target_workspace_identifier = '__TARGET_WORKSPACE_IDENTIFIER__'
target_lakehouse_name = '__TARGET_LAKEHOUSE_NAME__'

manifest = json.loads(fabric_fs.head(backup_manifest_path, 10 * 1024 * 1024))
target_root = f'abfss://{target_workspace_identifier}@onelake.dfs.fabric.microsoft.com/{target_lakehouse_name}.Lakehouse'

print(json.dumps(manifest, indent=2))
print(f'Target root: {target_root}')

In [ ]:
entries = manifest['backup']['entries']

for entry in entries:
    if entry['name'] == 'tables':
        restore_target = f'{target_root}/Tables'
    elif entry['name'] == 'files':
        restore_target = f'{target_root}/Files'
    elif entry['name'] == 'warehouse-export':
        print('Warehouse export restored to staging only; load into the DR warehouse separately using the warehouse recovery flow.')
        restore_target = f'{target_root}/Files/nonpaired-warehouse-staging'
    else:
        raise ValueError(f"Unexpected manifest entry: {entry['name']}")

    print(f"Restoring {entry['name']} from {entry['target']} -> {restore_target}")
    fabric_fs.cp(entry['target'], restore_target, True)

print('Restore copy complete')

In [ ]:
print('Next steps:')
print('1. Validate the lakehouse tables and files in the recovered workspace.')
print('2. If a warehouse export exists, use the warehouse recovery workflow to ingest it into the recovered warehouse.')
print('3. Continue with Step 4 post-recovery rebinding and validation.')